In [1]:
import asyncio
import httpx
import logging
import json
import sys
import os
from pathlib import Path
import time

In [4]:
import requests

In [71]:
from autogen_core.code_executor import CodeBlock,CodeExecutor
from autogen_ext.code_executors.docker import DockerCommandLineCodeExecutor
from autogen_ext.code_executors.local import LocalCommandLineCodeExecutor
from autogen_ext.code_executors.jupyter import JupyterCodeExecutor
import tempfile
from pathlib import Path
import venv
import asyncio
from autogen_core import CancellationToken
async def LocalCodeExecutor(codeblock_list,env=None,filedir='/oper/ch/autogen'):
    work_dir = Path(filedir)
    work_dir.mkdir(exist_ok=True)
    if not env:
        venv_dir = work_dir / ".venv"
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_builder.create(venv_dir)
        venv_context = venv_builder.ensure_directories(venv_dir)
    else:
        venv_builder = venv.EnvBuilder(with_pip=True)
        venv_context = venv_builder.ensure_directories(env)
    local_executor = LocalCommandLineCodeExecutor(work_dir=work_dir, virtual_env_context=venv_context)
    try:
        result = await local_executor.execute_code_blocks(
            code_blocks=codeblock_list,
            cancellation_token=CancellationToken(),)
        return result.output
    except Exception as e:
        return f"错误问题: {e}"

In [3]:
#sys.path.append('/oper/work/endian/intelligent_agent')

In [165]:
user_api_key={"api-key": "chenhao"}#hao

In [383]:
user_api_key={"api-key": "wangendian"}#endian

In [384]:
response = requests.post(
    "http://localhost:8005/tabs",
    headers=user_api_key,
    json={"provider": "claude"}
)
# 获取tab_id用于后续操作
tab_id = response.json()#["tab_id"]
print(tab_id)

{'status': 'success', 'tab_id': 'de0a4d9d-b449-4efc-8e6b-bff56d546f75', 'provider': 'claude', 'title': 'Claude', 'url': 'https://claude.ai/'}


In [196]:
response = requests.post(
    "http://localhost:8005/tabs/chatgpt/screenshot",
    headers=user_api_key,
    json={"provider": "chatgpt"}
)

In [197]:
response.json()

{'status': 'success',
 'screenshot_path': 'screenshots/screenshot_1743157130.png'}

In [45]:
def send_wechat_message():
    processed_params = {
        "contact_name": "陈浩",
        "message": "测试微信数据接口 "
    }
    response = requests.post(
        "http://localhost:8003/tools/wechat/search_and_send",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [412]:
def check_tax():
    processed_params = {
        "city": "深圳",
        "start_date": "2025-01-01",
        "end_date": "2025-04-01"
    }
    response = requests.post(
        "http://localhost:8003/tools/tax/navigate_to_main",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [436]:
response=check_tax()
print(response)

{'status': 'success', 'message': '成功打开纳税记录开具页面', 'tab_id': 2, 'url': 'https://its.shenzhen.chinatax.gov.cn:4433/gkpt/#/taxChecklist'}


In [424]:
def check_ssn():
    processed_params = {
    }
    response = requests.post(
        "http://localhost:8003/tools/social_security/navigate_and_select_person",
        json=processed_params,
        timeout=300.0
    )
    return response.json()

In [452]:
response=check_ssn()
print(response)

{'status': 'success', 'message': '社保清单查询完成并已下载', 'element_index': 2, 'element_text': '下载', 'new_tab_created': False, 'click_result': {'status': 'success', 'message': '成功点击元素: 下载', 'element_index': 2, 'element_text': '下载', 'new_tab_created': False, 'new_tab_ids': [], 'url': 'https://sipub.sz.gov.cn/hspms/logon.do?code=pm01_qQ5_fycIRsW9iV5UJgAeTg&gdbstoken=null&method=gdCasCallback&sxbm=btnDoPrintSbcbzm###', 'title': '深圳市社会保险基金管理局-个人网上服务', 'elements_count': 6}, 'is_done': True, 'task_success': True}


In [150]:
# 发送消息给Claude
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": """简单描述这段代码llm_page_analyzer是做什么的？
""",
        "file_paths":["/oper/work/endian/intelligent_agent/page_analyzer/llm_page_analyzer.py",
                     ],
        "new_chat": True
    }
)
#print(response.json())

In [158]:
# 发送消息给Claude
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": """写个执行python代码的命令
""",
        "file_paths":None,
        "new_chat": True
    }
)
#print(response.json())

In [155]:
print(response.json())

{'status': 'error', 'message': 'Could not start new chat'}


In [406]:
response = requests.post(
    "http://localhost:8005/chat/claude",
    headers=user_api_key,
    json={
        "tab_id": tab_id,
        "prompt": "stop",
        "file_paths":None,
        "new_chat": False
    }
)

KeyboardInterrupt: 

In [205]:
response.json()['messages'][-1]['content']['codeBlocks'][-1]

{'code': 'const rect = el.getBoundingClientRect();\n                        return text.length > 50 && // Has substantial text\n                              rect.width > 100 && // Has reasonable width\n                              !el.querySelector(\'pre\') && // Not just code\n                              rect.height > 30; // Not too small\n                    });\n                \n                // Group them into likely conversation turns by analyzing text patterns\n                const allText = textElements.map(el => el.textContent.trim());\n                \n                // Try to identify user vs assistant messages\n                const userPatterns = [\n                    /^you:/i, /^user:/i, /^human:/i, \n                    /^\\s*[A-Z]\\s+/, // Single letter prefix (like Claude\'s "H")\n                    /edit$/i, // Edit button suffix\n                ];\n                \n                const assistantPatterns = [\n                    /^claude:/i, /^assistant:

In [88]:
codeblock_list=[CodeBlock(language=line['language'],code=line['code']) for line in response.json()['messages'][-3]['content']['codeBlocks'] if line['language'] in ['python','bash','sh'] ]

In [93]:
code_exe_result=await LocalCodeExecutor(codeblock_list,env='/oper/work/endian/LLM-Assistant/py310/',filedir='/oper/work/endian/intelligent_agent/page_analyzer')

In [94]:
code_exe_result

"Added to path: /oper/work/endian/intelligent_agent\nSuccessfully imported BrowserSession\n正在查找包含 chatgpt.com 的标签页...\n找到包含 chatgpt.com 的标签页: FB8A15F3DA00BE968FBA2E302E72714A\n当前所有标签页: ['FB8A15F3DA00BE968FBA2E302E72714A', '6B8FB9E909F569A8CAB0476D21BBD064']\n目标标签页句柄: FB8A15F3DA00BE968FBA2E302E72714A\n"

In [37]:
#获取所有标签页：
response = requests.get(
    "http://localhost:8005/tabs",
    headers=user_api_key
)
tabs = response.json()
print(tabs)

[{'tab_id': '46e2a78c-b426-4cac-b736-66cf713beeae', 'provider': 'claude', 'title': 'Quantum CUBO Code Example - Claude', 'url': 'https://claude.ai/chat/02512880-b168-425b-a9d1-0b95b44f092b'}]


In [26]:
import uuid

def generate_api_key():
    """生成一个随机的API密钥"""
    return str(uuid.uuid4())

# 示例使用
new_user_key = generate_api_key()

In [30]:
generate_api_key()

'f16ab7cb-c2d9-4f2a-b2a9-968a0df75385'